In [2]:
import boto3
from datetime import datetime, timezone
import pandas as pd

cloudwatch = boto3.client("cloudwatch", region_name="ap-southeast-1")

# simulated data collection 
start_time = datetime(2026, 4, 17, 9, 30, 0, tzinfo=timezone.utc)
end_time   = datetime(2026, 4, 18, 2, 55, 0, tzinfo=timezone.utc)

# example ec2 instance for metrics data
instance_id = "i-005c8dbd5f3fb9829"
host_name = "ip-172-31-40-240"


In [3]:
# get cloudwatch metrics
response = cloudwatch.get_metric_data(
    MetricDataQueries=[
        {"Id": "cpu", "MetricStat": { "Metric": {
                                                "Namespace": "AWS/EC2",
                                                "MetricName": "CPUUtilization",
                                                "Dimensions": [
                                                    {"Name": "InstanceId", "Value": instance_id}
                                                ]
                },"Period": 300, "Stat": "Average"
                # the average over 5 minutes, as 5 minute interval is collected
            }, "ReturnData": True},
        {"Id": "network_in", "MetricStat": { "Metric": {
                                                "Namespace": "AWS/EC2",
                                                "MetricName": "NetworkIn",
                                                "Dimensions": [
                                                    {"Name": "InstanceId", "Value": instance_id}
                                                ]
                }, "Period": 300, "Stat": "Sum"
            }, "ReturnData": True },

        {"Id": "network_out", "MetricStat": { "Metric": {
                                                "Namespace": "AWS/EC2",
                                                "MetricName": "NetworkOut",
                                                "Dimensions": [
                                                    {"Name": "InstanceId", "Value": instance_id}
                                                ]
                }, "Period": 300, "Stat": "Sum"
            }, "ReturnData": True },
        {"Id": "mem_used", "MetricStat": { "Metric": {
                                                "Namespace": "CWAgent",
                                                "MetricName": "mem_used_percent",
                                                "Dimensions": [
                                                    {"Name": "host", "Value": host_name}
                                                ]
                }, "Period": 300, "Stat": "Average"
            }, "ReturnData": True 
        }
    ],
    StartTime=start_time,
    EndTime=end_time
)

# simulated data collection 
start_time = datetime(2026, 4, 17, 9, 30, 0, tzinfo=timezone.utc)
end_time   = datetime(2026, 4, 18, 2, 55, 0, tzinfo=timezone.utc)


In [4]:
data = {}

# loop through all metrics (cpu , network in, network out, memory)
for result in response["MetricDataResults"]:
    # loop through each timestamp and values, insert into dictionary 
    # only if timestamp does not exist to ensure primary key constraint
    for timestamp, values in zip(result["Timestamps"], result["Values"]):
        if timestamp not in data:
            data[timestamp] = {}
        data[timestamp][result["Id"]] = values


In [9]:
df = pd.DataFrame.from_dict(data, orient="index")
df.sort_index(inplace=True)

df

,cpu,network_in,network_out,mem_used
2026-04-17 17:30:00+08:00,0.511661,33709.0,37626.0,28.644464
2026-04-17 17:35:00+08:00,0.506665,33865.0,37158.0,28.626806
2026-04-17 17:40:00+08:00,0.506662,33733.0,36961.0,28.624106
2026-04-17 17:45:00+08:00,0.506752,40088.0,40482.0,28.621192
2026-04-17 17:50:00+08:00,0.509909,33775.0,36922.0,28.618192
...,...,...,...,...
2026-04-18 10:30:00+08:00,0.503245,33643.0,37456.0,29.202381
2026-04-18 10:35:00+08:00,0.505002,33757.0,37559.0,29.199167
2026-04-18 10:40:00+08:00,0.508336,47005.0,48462.0,29.196253
2026-04-18 10:45:00+08:00,0.500002,33511.0,37457.0,29.220167


In [42]:
import boto3

# Create an EC2 client object
ec2_client = boto3.client('ec2')

all_instances = []
ec2_response = ec2_client.describe_instances()

for reservation in ec2_response['Reservations']:
    for instance in reservation['Instances']:
        instance_id = instance['InstanceId']
        host_name = f"ip-{instance['PrivateIpAddress'].replace('.','-')}"
        all_instances.append((instance_id, host_name))



In [ ]:
for instance, host in all_instances:
    pass

i-005c8dbd5f3fb9829 ip-172-31-40-240
i-03b6a47dbfc54da39 ip-172-31-33-28


In [ ]:
cursor = conn.cursor()

cursor.execute("SELECT * FROM metrics;")

data = cursor.fetchall()
for i in data:
    print(i)


conn.close()
cursor.close()

# transform data and insert into database here 
